# Colab runner

Connect a Colab GPU runtime (T4 or better). Run the setup cell first in every session, then the stage you want. The setup cell clones the GitHub repo into `/content`; checkpoints, the data cache and `results/` (logs in `results/logs/`, plots in `results/figures/`) are stored in `MyDrive/rlhf/`.

If the runtime disconnects, run setup again and re-run the same stage: it resumes from its last save (every 250 steps).

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

!pip install -q datasets rouge-score matplotlib

repo = Path("/content/Summarization-from-feedback")
if not (repo / "train_sft.py").is_file():
    !git clone https://github.com/Areesha-008/Summarization-from-feedback.git /content/Summarization-from-feedback
else:
    !git -C /content/Summarization-from-feedback pull
os.chdir(repo)
root = Path.cwd()

drive_root = Path("/content/drive/MyDrive/rlhf")
for name in ("checkpoints", "data_cache", "results"):
    target = drive_root / name
    target.mkdir(parents=True, exist_ok=True)
    link = root / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        for path in link.rglob("*"):
            if path.is_file():
                dest = target / path.relative_to(link)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(path, dest)
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)

print("project:", root)
print("data cache:", (root / "data_cache" / "data.pkl").exists())
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
!nvidia-smi -L


## Stage 1 - SFT

In [ ]:
!python train_sft.py


In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("results/sft_samples.md", encoding="utf-8").read()))
display(Image("results/figures/fig1_sft_loss.png"))


## Stage 2 - Reward model

Needs `checkpoints/sft/` from Stage 1.

In [ ]:
!python train_reward.py

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/logs/reward_log.csv"))
display(Image("results/figures/fig2_reward.png"))

## Stage 3 - PPO

Needs `checkpoints/sft/` and `checkpoints/reward/` (Stages 1 and 2). If it runs out of GPU memory, set `N_CHUNKS = 2` in `train_ppo.py`.

In [ ]:
!python train_ppo.py

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/logs/ppo_log.csv").tail(10))
display(Image("results/figures/fig3_ppo.png"))

## Stage 4 - DPO

Needs `checkpoints/sft/` from Stage 1.

In [ ]:
!python train_dpo.py

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/logs/dpo_log.csv"))
display(Image("results/figures/fig4_dpo.png"))